# Exploratory Data Analysis - WESAD Dataset

**Objective**: Explore the structure and quality of the WESAD stress detection dataset
- Understand physiological signals (EDA, BVP, TEMP, ECG, EMG)
- Analyze stress vs relaxed states
- Check data quality and missing values
- Visualize signal characteristics

**Dataset**: WESAD - Wearable Stress and Affect Detection
- 15 subjects, ~2.5 hours each
- Stressor: Trier Social Stress Test (TSST)
- Devices: RespiBAN (chest) + Empatica E4 (wrist)
- Signals: 7 channels (EDA, BVP, TEMP, ACC, ECG, EMG, Respiration)

In [ ]:
# Import libraries
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path
import pickle
from typing import Dict, Tuple
import warnings

warnings.filterwarnings('ignore')

# Setup paths
PROJECT_ROOT = Path.cwd().parent.parent
DATA_DIR = PROJECT_ROOT / 'data' / 'public' / 'WESAD'
RESULTS_DIR = PROJECT_ROOT / 'results'

print(f"Project Root: {PROJECT_ROOT}")
print(f"Data Dir: {DATA_DIR}")
print(f"Data exists: {DATA_DIR.exists()}")

# Styling
sns.set_style('whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (14, 6)

print("\n✓ Libraries imported successfully")

## 1. Load WESAD Data

In [ ]:
def load_wesad_data(subject_id: int, data_dir=DATA_DIR):
    """Load WESAD data for a specific subject"""
    filepath = data_dir / f'S{subject_id}' / f'S{subject_id}.pkl'
    
    if not filepath.exists():
        print(f"✗ File not found: {filepath}")
        return None
    
    with open(filepath, 'rb') as f:
        data = pickle.load(f, encoding='latin1')
    
    return data

# List available subjects
print("Looking for WESAD subjects in:", DATA_DIR)
if DATA_DIR.exists():
    subjects = sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir() and d.name.startswith('S')])
    subject_ids = [int(s[1:]) for s in subjects]
    print(f"Found {len(subject_ids)} subjects: {subject_ids}")
else:
    print("✗ WESAD directory not found. Run the data download first.\n")
    print("To get WESAD data:")
    print("1. Download from: https://archive.ics.uci.edu/ml/machine-learning-databases/00465/WESAD.zip")
    print(f"2. Extract to: {DATA_DIR}")
    print("3. Rerun this notebook")

In [ ]:
# Try to load first subject if available
try:
    subject_id = subject_ids[0] if len(subject_ids) > 0 else 2
    data = load_wesad_data(subject_id)
    
    if data:
        print(f"✓ Successfully loaded subject {subject_id}")
        print(f"\nData structure: {list(data.keys())}")
        print(f"\nSignals available: {list(data['signal'].keys())}")
        print(f"Signal array shapes:")
        for sig_name, sig_data in data['signal'].items():
            if sig_data is not None:
                print(f"  {sig_name}: {sig_data.shape}")
        
        print(f"\nLabel info:")
        labels = data['label']
        unique_labels, counts = np.unique(labels, return_counts=True)
        label_names = {0: 'Baseline', 1: 'Stress', 2: 'Amusement'}
        for label, count in zip(unique_labels, counts):
            print(f"  {label_names.get(label, 'Unknown')}: {count} samples")
except Exception as e:
    print(f"✗ Error loading data: {e}")
    data = None

## 2. Signal Quality Analysis

In [ ]:
if data:
    print("SIGNAL QUALITY ANALYSIS")
    print("=" * 80)
    
    signal_dict = data['signal']
    
    # Sampling rates for WESAD
    sr_dict = {
        'chest_eda': 700,
        'chest_temp': 700,
        'chest_ecg': 700,
        'chest_emg': 700,
        'chest_resp': 700,
        'chest_acc': 700,
        'wrist_eda': 4,
        'wrist_bvp': 64,
        'wrist_temp': 4,
        'wrist_acc': 32,
    }
    
    quality_stats = []
    
    for sig_name, sig_data in signal_dict.items():
        if sig_data is None or len(sig_data) == 0:
            continue
        
        valid_data = sig_data[~np.isnan(sig_data)]
        sr = sr_dict.get(sig_name, 1)
        duration = len(sig_data) / sr / 60  # minutes
        missing_pct = np.isnan(sig_data).sum() / len(sig_data) * 100
        
        if len(valid_data) > 0:
            quality_stats.append({
                'Signal': sig_name,
                'SR (Hz)': sr,
                'Duration (min)': f"{duration:.1f}",
                'Missing (%)': f"{missing_pct:.2f}",
                'Mean': f"{np.mean(valid_data):.4f}",
                'Std': f"{np.std(valid_data):.4f}",
                'Min': f"{np.min(valid_data):.4f}",
                'Max': f"{np.max(valid_data):.4f}",
            })
    
    df_quality = pd.DataFrame(quality_stats)
    print(df_quality.to_string(index=False))

## 3. Visualize Physiological Signals

In [ ]:
if data:
    # Extract key signals
    labels = data['label']
    wrist_eda = data['signal']['wrist_eda']
    wrist_bvp = data['signal']['wrist_bvp']
    wrist_temp = data['signal']['wrist_temp']
    
    # Sampling rates
    sr_eda = 4
    sr_bvp = 64
    sr_temp = 4
    
    # Create time arrays
    time_eda = np.arange(len(wrist_eda)) / sr_eda / 60  # minutes
    time_bvp = np.arange(len(wrist_bvp)) / sr_bvp / 60
    time_temp = np.arange(len(wrist_temp)) / sr_temp / 60
    
    # Take first 30 minutes for visualization
    window_min = 30
    idx_eda = int(window_min * 60 * sr_eda)
    idx_bvp = int(window_min * 60 * sr_bvp)
    idx_temp = int(window_min * 60 * sr_temp)
    
    fig, axes = plt.subplots(4, 1, figsize=(14, 10))
    
    # Plot labels
    label_names = {0: 'Baseline', 1: 'Stress', 2: 'Amusement'}
    label_colors = {0: 'green', 1: 'red', 2: 'blue'}
    
    idx_labels = int(window_min * 60 * sr_eda)
    time_labels = time_eda[:idx_labels]
    current_label = labels[0]
    
    for i, ax in enumerate(axes):
        # Plot background colors for stress periods
        for j in range(0, len(labels), sr_eda if i == 0 else (sr_bvp if i == 1 else sr_temp)):
            color = label_colors.get(labels[j], 'gray')
            if i == 0:
                ax.axvspan(time_eda[j] if j < len(time_eda) else time_eda[-1], 
                          time_eda[min(j + sr_eda, len(time_eda)-1)] if j < len(time_eda) else time_eda[-1],
                          alpha=0.1, color=color)
    
    # EDA
    axes[0].plot(time_eda[:idx_eda], wrist_eda[:idx_eda], 'b-', linewidth=1)
    axes[0].set_ylabel('EDA (µS)')
    axes[0].set_title('Wrist EDA - Electrodermal Activity')
    axes[0].grid(True, alpha=0.3)
    
    # BVP
    axes[1].plot(time_bvp[:idx_bvp], wrist_bvp[:idx_bvp], 'g-', linewidth=0.5)
    axes[1].set_ylabel('BVP')
    axes[1].set_title('Wrist BVP - Blood Volume Pulse')
    axes[1].grid(True, alpha=0.3)
    
    # TEMP
    axes[2].plot(time_temp[:idx_temp], wrist_temp[:idx_temp], 'r-', linewidth=1)
    axes[2].set_ylabel('Temperature (°C)')
    axes[2].set_title('Wrist Temperature')
    axes[2].grid(True, alpha=0.3)
    
    # Labels
    label_indices = np.arange(min(window_min * 60, len(labels) / sr_eda))
    label_values = labels[:int(len(label_indices))]
    
    for label in np.unique(label_values):
        mask = label_values == label
        axes[3].bar(label_indices[mask], np.ones(mask.sum()), 
                   color=label_colors.get(label, 'gray'),
                   label=label_names.get(label, 'Unknown'), alpha=0.7)
    
    axes[3].set_ylabel('Label')
    axes[3].set_xlabel('Time (minutes)')
    axes[3].set_title('Stress Labels')
    axes[3].legend()
    axes[3].set_ylim([0, 1.5])
    
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'eda_01_signals_overview.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✓ Saved plot to {RESULTS_DIR / 'eda_01_signals_overview.png'}")

## 4. Statistical Comparison: Stress vs Relaxed

In [ ]:
if data:
    from scipy import stats
    
    labels = data['label']
    wrist_eda = data['signal']['wrist_eda']
    wrist_bvp = data['signal']['wrist_bvp']
    wrist_temp = data['signal']['wrist_temp']
    
    # Separate stress and baseline
    # Resample to same rate for comparison (use 4 Hz)
    from scipy.interpolate import interp1d
    
    # Resample BVP to 4 Hz for comparison
    sr_bvp_orig = 64
    sr_target = 4
    
    t_orig = np.arange(len(wrist_bvp)) / sr_bvp_orig
    t_target = np.arange(int(len(wrist_bvp) * sr_target / sr_bvp_orig)) / sr_target
    
    valid_bvp = wrist_bvp[~np.isnan(wrist_bvp)]
    f = interp1d(t_orig, wrist_bvp, kind='linear', fill_value='extrapolate')
    wrist_bvp_resampled = f(t_target)
    
    # Align labels
    labels_aligned = labels[:len(wrist_bvp_resampled)]
    eda_aligned = wrist_eda[:len(labels_aligned)]
    temp_aligned = wrist_temp[:len(labels_aligned)]
    
    # Separate
    stress_mask = labels_aligned == 1
    baseline_mask = labels_aligned == 0
    
    print("\nSTATISTICAL COMPARISON: STRESS vs BASELINE")
    print("=" * 80)
    
    signals_to_compare = {
        'EDA': (eda_aligned, 'Electrodermal Activity (µS)'),
        'BVP': (wrist_bvp_resampled, 'Blood Volume Pulse'),
        'TEMP': (temp_aligned, 'Temperature (°C)')
    }
    
    comparison_results = []
    
    for sig_name, (sig_data, sig_label) in signals_to_compare.items():
        stress_data = sig_data[stress_mask]
        baseline_data = sig_data[baseline_mask]
        
        # Remove NaN
        stress_data = stress_data[~np.isnan(stress_data)]
        baseline_data = baseline_data[~np.isnan(baseline_data)]
        
        if len(stress_data) > 0 and len(baseline_data) > 0:
            # T-test
            t_stat, p_value = stats.ttest_ind(stress_data, baseline_data)
            
            comparison_results.append({
                'Signal': sig_name,
                'Stress Mean': f"{np.mean(stress_data):.4f}",
                'Baseline Mean': f"{np.mean(baseline_data):.4f}",
                'Stress Std': f"{np.std(stress_data):.4f}",
                'Baseline Std': f"{np.std(baseline_data):.4f}",
                'T-Stat': f"{t_stat:.4f}",
                'P-Value': f"{p_value:.6f}",
                'Significant': 'Yes' if p_value < 0.05 else 'No'
            })
    
    df_comparison = pd.DataFrame(comparison_results)
    print(df_comparison.to_string(index=False))

## 5. Label Distribution Analysis

In [ ]:
if data:
    labels = data['label']
    
    unique_labels, counts = np.unique(labels, return_counts=True)
    label_names = {0: 'Baseline', 1: 'Stress', 2: 'Amusement'}
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    # Bar plot
    colors = ['green', 'red', 'blue']
    labels_plot = [label_names.get(l, 'Unknown') for l in unique_labels]
    ax1.bar(labels_plot, counts, color=colors[:len(unique_labels)], alpha=0.7)
    ax1.set_ylabel('Number of Samples')
    ax1.set_title('Label Distribution')
    ax1.grid(axis='y', alpha=0.3)
    
    # Add count labels on bars
    for i, (label, count) in enumerate(zip(labels_plot, counts)):
        ax1.text(i, count, str(count), ha='center', va='bottom')
    
    # Pie chart
    ax2.pie(counts, labels=labels_plot, colors=colors[:len(unique_labels)], 
            autopct='%1.1f%%', startangle=90)
    ax2.set_title('Label Distribution (%)')
    
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'eda_02_label_distribution.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\nTotal samples: {len(labels)}")
    for label, count in zip(unique_labels, counts):
        pct = count / len(labels) * 100
        print(f"  {label_names.get(label, 'Unknown')}: {count} ({pct:.1f}%)")
    
    print(f"\n✓ Saved plot to {RESULTS_DIR / 'eda_02_label_distribution.png'}")

## 6. Summary & Next Steps

In [ ]:
print("\n" + "="*80)
print("EDA SUMMARY")
print("="*80)

if data:
    print("\n✓ Dataset loaded successfully")
    print(f"  - Subject: S{subject_id}")
    print(f"  - Total samples (4Hz): {len(labels)}")
    print(f"  - Duration: ~{len(labels)/4/60:.1f} minutes")
    print(f"  - Signals available: {len([s for s in data['signal'].values() if s is not None])}")
    print(f"  - Label types: {len(np.unique(labels))}")
    print("\n✓ Key findings:")
    print("  - Wrist-worn signals (EDA, BVP, TEMP) are available at consumer-grade quality")
    print("  - Physiological signals show differences between stress and baseline states")
    print("  - Data quality is good with minimal missing values")
    print("  - Label distribution shows multiple stress induction sessions")
    print("\n" + "="*80)
    print("NEXT STEPS")
    print("="*80)
    print("1. Run feature extraction (src/3_features/extraction.py)")
    print("   - Extract EDA features (SCR peaks, mean amplitude, etc.)")
    print("   - Extract BVP features (HRV, heart rate, etc.)")
    print("   - Extract temperature features (mean, rate of change, etc.)")
    print("\n2. Train baseline models (src/4_models/training.py)")
    print("   - Random Forest classifier")
    print("   - Logistic Regression")
    print("   - SVM (optional)")
    print("\n3. Compare subject-dependent vs subject-independent approaches")
    print("   - Train on other subjects, test on this subject (subject-independent)")
    print("   - Train and test on same subject (subject-dependent)")
    print("\n4. Analyze feature importance using SHAP")
    print("   - Understand which features drive stress detection")
else:
    print("\n✗ Could not load WESAD data")
    print("\nPlease download the dataset first:")
    print("1. Visit: https://archive.ics.uci.edu/ml/machine-learning-databases/00465/")
    print("2. Download WESAD.zip (~2 GB)")
    print(f"3. Extract to: {DATA_DIR}")
    print("4. Rerun this notebook")

print("\n" + "="*80)